# Pipeline Bronze → Silver — Dataset Marcobre

## Objetivo
Este notebook implementa el pipeline de transformación de datos desde la capa **Bronze** hacia **Silver**, asegurando limpieza, estandarización temporal y cálculo de métricas operacionales necesarias para reporting y analítica.

El proceso está diseñado para ejecutarse en Databricks y utiliza Apache Spark como motor de procesamiento distribuido.

---

## Flujo del procesamiento

### 1 Lectura desde Bronze
- Ingesta del dataset maestro operacional desde la capa Bronze.

### 2 Limpieza de datos
- Conversión de valores `NaN` a `NULL`.
- Estandarización de tipos numéricos para evitar errores en cálculos.

### 3 Estandarización temporal
- Generación de `DateKey` (formato YYYYMMDD).
- Generación de `YearMonth` para cálculos mensuales.

### 4 Cálculos operacionales
- Acumulados mensuales por columnas seleccionadas.
- Replicación de valores mensuales de planificación.
- Ventanas móviles de 15 días.

### 5 Carga a Silver
- Escritura en formato Delta.
- Sobrescritura controlada del esquema.

---

## Tabla destino
`dev_silver_curated_minsur.marcobre_fileserver.dataset_master`



In [0]:
%run ./Parametros

In [0]:
TABLE_BRONZE = f"{bronze_env}.marcobre_fileserver.dataset_master"
TABLE_SILVER = f"{silver_env}.marcobre_fileserver.dataset_master"

In [0]:
df = spark.read.table(TABLE_BRONZE)

df_clean = df
for field in df.schema.fields:
    if isinstance(field.dataType, (DoubleType, FloatType)):
        df_clean = df_clean.withColumn(
            field.name,
            F.when(F.isnan(F.col(field.name)), F.lit(None)).otherwise(F.col(field.name))
        )

df_silver = (
    df_clean
    .withColumn("FechaTS", F.to_timestamp("fecha", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("DateKey", F.date_format("FechaTS", "yyyyMMdd").cast("int"))
    .withColumn("YearMonth", F.date_format("FechaTS", "yyyyMM").cast("int"))
)

ColumnasAcum = [
    "consumo_de_acido_consumo_planta_oxs",
    "chancado_terciario_faja_cv_311"
]

w_acum = (
    Window
    .partitionBy("YearMonth")
    .orderBy("DateKey")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

for col in ColumnasAcum:
    df_silver = df_silver.withColumn(
        f"{col}_acumulado",
        F.sum(F.coalesce(F.col(col), F.lit(0))).over(w_acum)
    )

In [0]:
ColumnasInterpoladas = [
"volumen_minimo_x_proteccion_anodica_total",
"tonelaje_ingresado_a_bateas_plan_total_mes",
"consumo_de_acido_total_plan_total_mes",
"volumen_agua_de_mar_consumida_plan_total_mes",
"throughput_alimento_molino_plan_total_mes",
"throughput_carga_bateas_plan_total_mes",
"ley_de_cusac_cabeza_plan_total_mes",
"pctc_cabeza_plan_total_mes",
"consumo_de_acido_tambor_plan_total_mes",
"consumo_de_acido_faja_cv_312_plan_total_mes",
"throughput_descarga_bateas_plan_total_mes",
"flujo_de_pls_tratado_hg_lg_plan_total_mes",
"cu_en_pls_hg_a_sx_e1_e2_plan_total_mes",
"cu_en_pls_lg_a_sx_ep_plan_total_mes",
"eficiencia_extraccion_sx_plan_total_mes",
"eficiencia_sx_plan_total_mes",
"flujo_de_electrolito_comercial_plan_total_mes",
"cu_en_electrolito_rico_plan_total_mes",
"cu_en_electrolito_pobre_plan_total_mes",
"recuperacion_total_cusac_oxidos_plan_total_mes",
"stock_de_acido_puerto_y_site_plan",
"stock_de_agua_de_mar_plan"
]

w_month = Window.partitionBy("YearMonth")

for col in ColumnasInterpoladas:
    df_silver = df_silver.withColumn(
        f"{col}_interpolado",
        F.sum(F.coalesce(F.col(col), F.lit(0))).over(w_month)
    )

In [0]:
ColumnasRolling15 = [
    "catodos_cu_cosecha",
    "cu_total_transferido_ew"
]

w_15d = (
    Window
    .orderBy("DateKey")
    .rowsBetween(-14, 0)
)

for col in ColumnasRolling15:
    df_silver = df_silver.withColumn(
        f"{col}_rolling_15d",
        F.sum(F.coalesce(F.col(col), F.lit(0))).over(w_15d)
    )
    
df_silver = df_silver.withColumn(
    "eficiencia_electrodeposicion",
    F.expr("""
        try_divide(
            catodos_cu_cosecha_rolling_15d,
            cu_total_transferido_ew_rolling_15d
        )
    """)
)

df_silver = df_silver.drop(
    "catodos_cu_cosecha_rolling_15d",
    "cu_total_transferido_ew_rolling_15d"
)

In [0]:
df_silver = df_silver.drop("FechaTS", "YearMonth")

cols = ["DateKey"] + [c for c in df_silver.columns if c != "DateKey"]
df_silver = df_silver.select(cols)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_SILVER)